In [114]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import joblib
from sklearn.metrics import precision_recall_curve, auc
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.optimizers import Adam, SGD
from keras.models import load_model
import xgboost as xgb
import lightgbm as lgb
import time

In [116]:
def prepare_data(data):
    X = data.drop(['sample', 'regulator', 'target', 'interaction', 'size'], axis=1).values
    y = data['interaction'].values
    X = np.asarray(X).astype(np.float32)
    y = np.asarray(y).astype(np.float32)
    return X, y

In [118]:
def load_size10(limit=None):
    training_set = pd.read_csv(r'./caocao/training/size10.csv', index_col=0)
    if limit is not None:
        row_limit = limit*10*9
        training_set = training_set.head(row_limit).copy()
    valid_set = pd.read_csv(r'./caocao/validation/size10.csv', index_col=0)
    X_train, y_train = prepare_data(training_set)
    X_valid, y_valid = prepare_data(valid_set)
    del training_set
    del valid_set
    return X_train, y_train, X_valid, y_valid

In [76]:
def get_test_set(size):
    test_set_report = pd.read_csv(fr'./jump3_code/data for comparison/size{size}/size{size}.csv', index_col=0)
    X_test_report, y_test_report = prepare_data(test_set_report)
    del test_set_report
    return X_test_report, y_test_report

In [78]:
def get_report(model,report_filename, size, sample_number_max):
    edges = size*(size-1)
    start_predition_time = time.time()
    y_test_report_pred = model.predict(X_test)
    end_predition_time = time.time()
    with open(report_filename, 'w+') as f:
        for i in range(0, edges*sample_number_max*2, edges):
            precision1, recall1, _ = precision_recall_curve(y_test[i:i+edges], y_test_report_pred[i:i+edges])
            aupr1 = auc(recall1, precision1)
            f.write(f'{aupr1}\n')         
    return (end_predition_time-start_predition_time)*1000  

In [80]:
def get_report_xgb(model,report_filename, size, sample_number_max):
    edges = size*(size-1)
    dval = xgb.DMatrix(X_test, label=y_test)
    start_predition_time = time.time()
    y_test_report_pred = model.predict(dval)
    end_predition_time = time.time()
    with open(report_filename, 'w+') as f:
        for i in range(0, edges*sample_number_max*2, edges):
            precision1, recall1, _ = precision_recall_curve(y_test[i:i+edges], y_test_report_pred[i:i+edges])
            aupr1 = auc(recall1, precision1)
            f.write(f'{aupr1}\n')           
    return (end_predition_time-start_predition_time)*1000  

In [82]:
def get_report_lgb(model, report_filename, size, sample_number_max):
    edges = size*(size-1)
    start_predition_time = time.time()
    y_test_report_pred = model.predict(X_test, num_iteration=model.best_iteration)
    end_predition_time = time.time()
    with open(report_filename, 'w+') as f:
        for i in range(0, edges*sample_number_max*2, edges):
            precision1, recall1, _ = precision_recall_curve(y_test[i:i+edges], y_test_report_pred[i:i+edges])
            aupr1 = auc(recall1, precision1)
            f.write(f'{aupr1}\n')            
    return (end_predition_time-start_predition_time)*1000  

In [84]:
def run(folder, size):
    if size==10:
        sample_number_max = 10
    else:
        sample_number_max = 4
    with open(f'./caocao/{folder}/times_{size}.txt', 'w+') as f:        
        ##############################################################
        print(f'lr size {size} ...')
        lin_reg = LinearRegression()
        lr_start_training_time = time.time()
        lin_reg.fit(X_train, y_train)
        lr_end_training_time = time.time()
        f.write(f'{(lr_end_training_time - lr_start_training_time)*1000}\n')
        lr_prediction_time = get_report(lin_reg, f'./caocao/{folder}/linear_size{size}.txt', size, sample_number_max)
        f.write(f'{lr_prediction_time}\n')
        joblib.dump(lin_reg, fr'./caocao/{folder}/models/size{size}_ln.pkl')
        del lin_reg
        ##############################################################
        print(f'dt size {size}')
        dt_model = DecisionTreeClassifier(random_state=42)
        dt_start_training_time = time.time()
        dt_model.fit(X_train, y_train)
        dt_end_training_time = time.time()
        f.write(f'{(dt_end_training_time - dt_start_training_time)*1000}\n')
        #dt_prediction_time = get_report(dt_model, f'./caocao/report_update/model{model_size}/dt_size50.txt', 50, 4)
        #dt_prediction_time = get_report(dt_model, f'./caocao/report_update/model{model_size}/dt_size10.txt', 10, 10)
        dt_prediction_time = get_report(dt_model, f'./caocao/{folder}/dt_size{size}.txt', size, sample_number_max)
        f.write(f'{dt_prediction_time}\n')
        joblib.dump(dt_model, fr'./caocao/{folder}/models/size{size}_dt.pkl')
        del dt_model
        ##############################################################
        print(f'rf size {size}')
        rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
        rf_start_training_time = time.time()
        rf_model.fit(X_train, y_train)
        rf_end_training_time = time.time()
        f.write(f'{(rf_end_training_time - rf_start_training_time)*1000}\n')
        #rf_prediction_time = get_report(rf_model, f'./caocao/report_update/model{model_size}/rf_size50.txt', 50, 4)
        #rf_prediction_time = get_report(rf_model, f'./caocao/report_update/model{model_size}/rf_size10.txt', 10, 10)
        rf_prediction_time = get_report(rf_model, f'./caocao/{folder}/rf_size{size}.txt', size, sample_number_max)
        f.write(f'{rf_prediction_time}\n')
        joblib.dump(rf_model, fr'./caocao/{folder}/models/size{size}_rf.pkl')
        del rf_model
        ##############################################################
        print(f'knn size {size}')
        knn_model = KNeighborsClassifier(n_neighbors=10)
        # k=10 is the best
        # Train the model
        knn_start_training_time = time.time()
        knn_model.fit(X_train, y_train)
        knn_end_training_time = time.time()
        f.write(f'{(knn_end_training_time - knn_start_training_time)*1000}\n')
        #knn_prediction_time = get_report(knn_model, f'./caocao/report_update/model{model_size}/knn_size50.txt', 50, 4)
        #knn_prediction_time = get_report(knn_model, f'./caocao/report_update/model{model_size}/knn_size10.txt', 10, 10)
        knn_prediction_time = get_report(knn_model, f'./caocao/{folder}/knn_size{size}.txt', size, sample_number_max)
        f.write(f'{knn_prediction_time}\n')
        joblib.dump(knn_model, fr'./caocao/{folder}/models/size{size}_knn.pkl')
        del knn_model
        ##############################################################
        print(f'xgb size {size}')
        dtrain = xgb.DMatrix(X_train, label=y_train)
        dval = xgb.DMatrix(X_valid, label=y_valid)
        params = {
            'max_depth': 3,         # Maximum depth of a tree
            'eta': 0.1,             # Learning rate
            'objective': 'binary:logistic',  # Binary classification objective
            'eval_metric': 'logloss' # Evaluation metric
        }
        evallist = [(dtrain, 'train'), (dval, 'eval')]
        num_round = 100  # Number of boosting rounds
        xgb_start_training_time = time.time()
        xgb_model = xgb.train(params, dtrain, num_round, evals=evallist, early_stopping_rounds=10)
        xgb_end_training_time = time.time()
        f.write(f'{(xgb_end_training_time - xgb_start_training_time)*1000}\n')
        #
        #xgb_prediction_time = get_report_xgb(xgb_model, f'./caocao/report_update/model{model_size}/xgb_size10.txt', 10, 10)    
        #xgb_prediction_time = get_report_xgb(xgb_model, f'./caocao/report_update/model{model_size}/xgb_size50.txt', 50, 4)
        xgb_prediction_time = get_report_xgb(xgb_model, f'./caocao/{folder}/xgb_size{size}.txt', size, sample_number_max)
        f.write(f'{xgb_prediction_time}\n')
        #xgb_prediction_time = xgb_model.save_model(fr'./caocao/{folder}/models/size{size}_xgb.pkl')
        xgb_model.save_model(fr'./caocao/{folder}/models/size{size}_xgb.json')
        del xgb_model
        ##############################################################
        print(f'lgb size {size}')
        train_data = lgb.Dataset(X_train, label=y_train)
        val_data = lgb.Dataset(X_valid, label=y_valid, reference=train_data)
        params = {
            'boosting_type': 'gbdt',  # Gradient Boosting Decision Tree
            'objective': 'binary',    # Binary classification
            'metric': 'binary_logloss', # Metric to evaluate
            'num_leaves': 31,         # Maximum tree leaves for base learners
            'learning_rate': 0.05,    # Learning rate
            'feature_fraction': 0.9   # Fraction of features to be used for each tree
        }
        lgb_start_training_time = time.time()
        lgb_model = lgb.train(params, train_data, num_boost_round=100, valid_sets=[train_data, val_data])
        lgb_end_training_time = time.time()
        f.write(f'{(lgb_end_training_time - lgb_start_training_time)*1000}\n')
       # 
       # lgb_prediction_time = get_report_lgb(lgb_model, f'./caocao/report_update/model{model_size}/lgb_size10.txt', 10, 10)
        #lgb_prediction_time = get_report_lgb(lgb_model, f'./caocao/report_update/model{model_size}/lgb_size50.txt', 50, 4)
        lgb_prediction_time = get_report_lgb(lgb_model, f'./caocao/{folder}/lgb_size{size}.txt', size, sample_number_max)
        f.write(f'{lgb_prediction_time}\n')
        lgb_model.save_model(fr'./caocao/{folder}/models/size{size}_lgb.txt')
        del lgb_model
        ##############################################################
        print(f'nb size {size}')
        nb_classifier = GaussianNB()
        nb_start_training_time = time.time()
        nb_classifier.fit(X_train, y_train)
        nb_end_training_time = time.time()
        f.write(f'{(nb_end_training_time - nb_start_training_time)*1000}\n')
        #nb_prediction_time = get_report(nb_classifier, f'./caocao/report_update/model{model_size}/nb_size50.txt', 50, 4)
        #nb_prediction_time = get_report(nb_classifier, f'./caocao/report_update/model{model_size}/nb_size10.txt', 10, 10)
        nb_prediction_time = get_report(nb_classifier, f'./caocao/{folder}/nb_size{size}.txt', size, sample_number_max)
        f.write(f'{nb_prediction_time}\n')
        joblib.dump(nb_classifier, fr'./caocao/{folder}/models/size{size}__nb.pkl')
        del nb_classifier

In [128]:
X_train, y_train, X_valid, y_valid = load_size10(limit=None)

In [130]:
nb_classifier = GaussianNB()
nb_start_training_time = time.time()
nb_classifier.fit(X_train, y_train)
nb_end_training_time = time.time()
print(f'{(nb_end_training_time - nb_start_training_time)*1000}\n')

1590.0349617004395



In [86]:
def predict(experiment_folder, size, sample_number_max):
    with open(f'./caocao/{experiment_folder}/test_size{size}/times_{size}.txt', 'w+') as f:   
        lin_reg = joblib.load(fr'./caocao/{experiment_folder}/models/size10_ln.pkl')
        lr_prediction_time = get_report(lin_reg, f'./caocao/{experiment_folder}/test_size{size}/linear_size{size}.txt', size, sample_number_max)
        f.write(f'{lr_prediction_time}\n')

        dt_model = joblib.load(fr'./caocao/{experiment_folder}/models/size10_dt.pkl')
        dt_prediction_time = get_report(dt_model, f'./caocao/{experiment_folder}/test_size{size}/dt_size{size}.txt', size, sample_number_max)
        f.write(f'{dt_prediction_time}\n')

        rf_model = joblib.load(fr'./caocao/{experiment_folder}/models/size10_rf.pkl')
        rf_prediction_time = get_report(rf_model, f'./caocao/{experiment_folder}/test_size{size}/rf_size{size}.txt', size, sample_number_max)
        f.write(f'{rf_prediction_time}\n')

        knn_model = joblib.load(fr'./caocao/{experiment_folder}/models/size10_knn.pkl')
        knn_prediction_time = get_report(knn_model, f'./caocao/{experiment_folder}/test_size{size}/knn_size{size}.txt', size, sample_number_max)
        f.write(f'{knn_prediction_time}\n')

        xgb_model = xgb.Booster()
        xgb_model.load_model(fr'./caocao/{experiment_folder}/models/size10_xgb.json')
        #xgb_model = joblib.load(fr'./caocao/{experiment_folder}/models/size10_xgb.pkl')
        xgb_prediction_time = get_report_xgb(xgb_model, f'./caocao/{experiment_folder}/test_size{size}/xgb_size{size}.txt', size, sample_number_max)
        f.write(f'{xgb_prediction_time}\n')

        lgb_model = lgb.Booster(model_file=fr'./caocao/{experiment_folder}/models/size10_lgb.txt')
        lgb_prediction_time = get_report_lgb(lgb_model, f'./caocao/{experiment_folder}/test_size{size}/lgb_size{size}.txt', size, sample_number_max)
        f.write(f'{lgb_prediction_time}\n')

        nb_classifier = joblib.load(fr'./caocao/{experiment_folder}/models/size10__nb.pkl')
        nb_prediction_time = get_report(nb_classifier, f'./caocao/{experiment_folder}/test_size{size}/nb_size{size}.txt', size, sample_number_max)
        f.write(f'{nb_prediction_time}\n')


In [68]:
X_test, y_test = get_test_set(10)
for limit in [5000, 10000]:
    folder = f'report_size10_{limit}'
    X_train, y_train, X_valid, y_valid = load_size10(limit=limit)
    run(folder, 10)

KeyboardInterrupt: 

In [94]:
experiment_folder = 'report_size10_5000'
X_test, y_test = get_test_set(50)
predict(experiment_folder, 50, 4)

In [106]:
X_train, y_train, X_valid, y_valid = load_size10(limit=10000)

In [108]:
X_train.shape

(900000, 102)

In [110]:
positive = 0
nav = 0
for i in y_train:
    if i==1:
        positive+=1
    else:
        nav+=1
print(positive, nav)

134600 765400


In [112]:
print(positive*100/(positive+nav))

14.955555555555556
